# MARL-Gated RALA ViT — Object Detection via MMDetection

**Self-contained notebook** that implements the full detection pipeline:

1. All modules defined inline (attention, backbone, reward, hooks)
2. MMDetection config built programmatically
3. Training with curriculum (router freeze → PPO)
4. Evaluation and visualization

---

### Comparison with Original RALA Repo (`github.com/qhfan/RALA`)

| Aspect | Original RAVLT-T | Our MARL-RALA |
|---|---|---|
| **Architecture** | 4-stage hierarchical `[2,2,6,2]` depths, `[64,128,256,512]` dims | Flat 16-layer ViT (d=256) + 4 FPN projection layers |
| **Backbone type** | `GLTA` (registered) | `MARLRALABackbone` (registered) |
| **α_j reweighting** | Attention-score based (learned) | MARL Actor-Critic routing `w ∈ (0,1)` per token |
| **Output gate** | Hadamard `φ(X) ⊙ Attn(X)` | **Matches RALA:** Fused QKVO projection + raw gate `O ⊙ (Attn(X) + LePE)` |
| **Kernel function** | `ELU(·)+1` | `ELU(·)+1` |
| **FPN in_channels** | `[64, 128, 256, 512]` (native per-stage) | `[256, 256, 256, 256]` (projected via deconv/conv) |
| **Detection heads** | RetinaNet 1×, Mask R-CNN 1×/3×, Cascade 3× | RetinaNet 1× (primary), Mask R-CNN (planned) |
| **Optimizer** | AdamW lr=1e-4, wd=1e-4 (RetinaNet) / wd=0.05 (MRCNN) | AdamW lr=1e-4, wd=0.05 |
| **Schedule** | 12 epochs, steps=[8,11], 500-iter warmup | Same |
| **Batch size** | 2/gpu | 2/gpu (configurable) |
| **FP16** | Yes (loss_scale=512) | Yes (native AMP) |
| **Budget/routing** | None (standard attention) | Entropy-based redundancy penalty + bimodal sparsity |
| **PPO** | None | Clipped surrogate PPO on router after each iter |
| **Training curriculum** | Standard | Freeze router epochs 0-3, unfreeze 4-11 |
| **Pretrained** | ImageNet-1K 300 epochs | ImageNet-100 Stage 2 (your checkpoint) |

---

### Key Hyperparameters from RALA Repo

```python
# From retinanet_t_1x.py:
embed_dims = [64, 128, 256, 512]  # hierarchical channels
depths     = [2, 2, 6, 2]         # blocks per stage
num_heads  = [1, 2, 4, 8]         # heads per stage
mlp_ratios = [3.5, 3.5, 3.5, 3.5]
projection = 1024                  # for RALA rank augmentation
optimizer  = AdamW(lr=1e-4, wd=1e-4)
schedule   = 12 epochs, steps=[8,11], warmup=500 iters
fp16       = True (loss_scale=512)
```

## 0. Install Dependencies

In [ ]:
# 1. Install openmim 
!pip install -U openmim --no-cache-dir

# 2. Patch setuptools for Python 3.12 compatibility
!pip install setuptools==69.5.1

# 3. THE FIX: Explicitly pin mmcv to version 2.1.0 to satisfy mmdet's assertion check
!mim install mmengine "mmcv==2.1.0" mmdet

# 4. Install specific dependencies
!pip install einops

## 1. Imports

In [1]:
import os, sys, math, warnings, logging
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from einops import rearrange
import numpy as np

# Check MMDetection availability
try:
    from mmengine.model import BaseModule
    from mmengine.config import Config
    from mmengine.runner import Runner
    from mmdet.registry import MODELS, HOOKS
    from mmdet.utils import setup_cache_size_limit_of_dynamo
    HAS_MMDET = True
    print("[OK] MMDetection found")
except ImportError:
    HAS_MMDET = False
    BaseModule = nn.Module
    print("[WARN] MMDetection not installed. Run: pip install -U openmim && mim install mmengine mmcv mmdet")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

[OK] MMDetection found
Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 2. Core Modules: Attention, Router, Encoder

In [3]:
class DropPath(nn.Module):
    """Drop paths (Stochastic Depth) per sample."""
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training or self.drop_prob == 0.0:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor = torch.floor(random_tensor + keep_prob)
        return x / keep_prob * random_tensor


class SharedActorCritic(nn.Module):
    """Shared Actor-Critic for the swarm of N agents (one per token)."""

    def __init__(self, d_model: int):
        super().__init__()
        half = max(d_model // 2, 16)
        self.proj = nn.Linear(d_model, half)
        self.actor_mlp = nn.Sequential(nn.Linear(half, half), nn.ELU())
        self.mu_head = nn.Linear(half, 1)
        self.sigma_head = nn.Linear(half, 1)
        self.critic_mlp = nn.Sequential(nn.Linear(half, half), nn.ELU(), nn.Linear(half, 1))

    def forward(self, local_features, deterministic=False):
        s = self.proj(local_features)
        h_actor = self.actor_mlp(s)
        mu = self.mu_head(h_actor).squeeze(-1)
        sigma = F.softplus(self.sigma_head(h_actor)).squeeze(-1) + 1e-5
        value = self.critic_mlp(s).squeeze(-1)

        if deterministic:
            z = mu
            log_prob = None
        else:
            dist = Normal(mu, sigma)
            z = dist.rsample()
            log_prob = dist.log_prob(z)

        w_raw = torch.tanh(z)
        w = (w_raw + 1.0) / 2.0

        if not deterministic:
            jacobian = torch.log(0.5 * (1.0 - w_raw.pow(2)) + 1e-5)
            log_prob = log_prob - jacobian

        return w, log_prob, value, mu, sigma


class ChunkwiseRALAAttention(nn.Module):
    """Chunkwise Linear Attention with MARL gating (replaces RALA's alpha_j)."""

    def __init__(self, d_model, head=8, chunk_size=16, gamma=0.1, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.head = head
        self.chunk_size = chunk_size
        self.gamma = gamma
        self.d_k = d_model // head
        # Fused Q/K/V/O projection (matches RALA's Conv2d(dim, dim*4, 1))
        self.qkvo = nn.Linear(d_model, d_model * 4)
        # LePE: 5x5 depthwise conv on V (matches RALA's lepe)
        self.lepe = nn.Conv2d(d_model, d_model, 5, 1, 2, groups=d_model)
        # Output projection (matches RALA's self.proj)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def _pad_to_chunk(self, x, w_gating):
        b, n, d = x.shape
        C = self.chunk_size
        remainder = n % C
        if remainder == 0:
            return x, w_gating, n
        pad_len = C - remainder
        x = F.pad(x, (0, 0, 0, pad_len))
        w_gating = F.pad(w_gating, (0, pad_len))
        return x, w_gating, n

    def forward(self, x, w_gating, H, W, use_dilution=False):
        x, w_gating, orig_n = self._pad_to_chunk(x, w_gating)
        b, n, d = x.shape
        T = n // self.chunk_size
        C = self.chunk_size

        qkvo = self.qkvo(x)                          # (B, N, 4*d_model)
        qkv = qkvo[:, :, :3 * self.d_model]           # (B, N, 3*d_model)
        o = qkvo[:, :, 3 * self.d_model:]              # (B, N, d_model) — output gate

        v_for_lepe = qkv[:, :orig_n, 2 * self.d_model:]  # (B, orig_n, d_model)
        lepe = v_for_lepe.transpose(1, 2).view(b, self.d_model, H, W)  # (B, C, H, W)
        lepe = self.lepe(lepe)                         # (B, C, H, W)
        lepe = lepe.view(b, self.d_model, -1).transpose(1, 2)  # (B, H*W, d_model)

        q, k, v = qkv.chunk(3, dim=-1)
        q = rearrange(q, 'b (T C) (h dk) -> b T h C dk', T=T, C=C, h=self.head)
        k = rearrange(k, 'b (T C) (h dk) -> b T h C dk', T=T, C=C, h=self.head)
        v = rearrange(v, 'b (T C) (h dk) -> b T h C dk', T=T, C=C, h=self.head)

        q = q * (self.d_k ** -0.25)
        k = k * (self.d_k ** -0.25)
        phi_q = F.elu(q) + 1.0
        phi_k = F.elu(k) + 1.0

        w_chunks = rearrange(w_gating, 'b (T C) -> b T C', T=T, C=C)
        w_expanded = w_chunks.unsqueeze(2).unsqueeze(-1)
        k_gated = w_expanded * phi_k

        k_gated_f32 = k_gated.to(torch.float32)
        v_f32 = v.to(torch.float32)
        KV_chunks = torch.matmul(k_gated_f32.transpose(-2, -1), v_f32)
        Z_chunks = k_gated_f32.sum(dim=-2)
        w_bar = w_chunks.mean(dim=-1)

        outputs = []
        S = torch.zeros(b, self.head, self.d_k, self.d_k, device=x.device, dtype=torch.float32)
        Z = torch.zeros(b, self.head, self.d_k, device=x.device, dtype=torch.float32)

        for t in range(T):
            decay_factor = 1.0 - (self.gamma * (1.0 - w_bar[:, t]))
            decay_S = decay_factor.view(b, 1, 1, 1)
            decay_Z = decay_factor.view(b, 1, 1)

            if use_dilution and t > 0:
                dilution_scale = self.gamma * (1.0 - w_bar[:, t])
                gamma_tau = dilution_scale.view(b, 1, 1, 1) * S / max(t, 1)
                S = (S * decay_S) + KV_chunks[:, t] + gamma_tau
            else:
                S = (S * decay_S) + KV_chunks[:, t]

            Z = (Z * decay_Z) + Z_chunks[:, t]
            phi_q_t = phi_q[:, t].to(torch.float32)
            nom = torch.matmul(phi_q_t, S)
            denom = (phi_q_t * Z.unsqueeze(-2)).sum(dim=-1, keepdim=True) + 1e-5
            out_t = nom / denom
            if self.training:
                out_t = self.dropout(out_t)
            out_t = torch.clamp(out_t, min=-65000.0, max=65000.0)
            outputs.append(out_t.to(q.dtype))

        out = torch.stack(outputs, dim=1)
        out = rearrange(out, 'b T h C dk -> b (T C) (h dk)')
        out = out[:, :orig_n, :]
        out = out + lepe
        o = o[:, :orig_n, :]
        out = self.proj(out * o)
        return out, phi_k


class MLP(nn.Module):
    def __init__(self, in_features, hidden_features, drop=0.):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, in_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class ConditionalPositionEncoding(nn.Module):
    """CPE: 3x3 depthwise conv with residual (from RAVLT/CPVT)."""
    def __init__(self, d_model):
        super().__init__()
        self.dwconv = nn.Conv2d(d_model, d_model, kernel_size=3, padding=1, groups=d_model)

    def forward(self, x, H, W):
        B, N, C = x.shape
        feat = x.transpose(1, 2).view(B, C, H, W)
        feat = self.dwconv(feat)
        feat = feat.view(B, C, N).transpose(1, 2)
        return x + feat


class EncoderBlock(nn.Module):
    def __init__(self, d_model, head, chunk_size, drop_path=0.0):
        super().__init__()
        self.cpe = ConditionalPositionEncoding(d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = ChunkwiseRALAAttention(d_model, head=head, chunk_size=chunk_size)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, d_model * 4)
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

    def forward(self, x, w_gating, H, W, use_dilution=False):
        x = self.cpe(x, H, W)
        res = x
        out, phi_k = self.attn(self.norm1(x), w_gating, H, W, use_dilution)
        x = res + self.drop_path(out)
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x, phi_k


class PatchEmbedding(nn.Module):
    def __init__(self, patch_size=16, in_chans=3, embed_dim=256):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        B, C, H_p, W_p = x.shape
        x = x.flatten(2).transpose(1, 2)
        return x, H_p, W_p


print("[OK] Core modules defined")

[OK] Core modules defined


## 3. Backbone: Flat ViT + FPN Projection Layers

In [25]:
class FPNProjection(nn.Module):
    """Projects flat ViT features into one FPN level via spatial rescaling."""

    def __init__(self, in_channels, out_channels, scale_factor):
        super().__init__()
        self.scale_factor = scale_factor
        layers = []
        if scale_factor == 2.0:
            layers += [nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2),
                       nn.BatchNorm2d(out_channels), nn.GELU()]
        elif scale_factor == 1.0:
            layers += [nn.Conv2d(in_channels, out_channels, 1),
                       nn.BatchNorm2d(out_channels), nn.GELU()]
        elif scale_factor == 0.5:
            layers += [nn.Conv2d(in_channels, out_channels, 3, stride=2, padding=1),
                       nn.BatchNorm2d(out_channels), nn.GELU()]
        elif scale_factor == 0.25:
            layers += [nn.Conv2d(in_channels, out_channels, 3, stride=2, padding=1),
                       nn.BatchNorm2d(out_channels), nn.GELU(),
                       nn.Conv2d(out_channels, out_channels, 3, stride=2, padding=1),
                       nn.BatchNorm2d(out_channels), nn.GELU()]
        self.proj = nn.Sequential(*layers)

    def forward(self, tokens, H, W):
        B, N, C = tokens.shape
        x = tokens.transpose(1, 2).view(B, C, H, W)
        return self.proj(x)


_base_cls = BaseModule if HAS_MMDET else nn.Module

class MARLRALABackbone(_base_cls):
    """
    Flat 16-layer MARL-Gated RALA ViT with FPN projection layers.
    
    Extracts features at 4 intermediate depths and projects them into
    a 4-level feature pyramid for FPN-based detection heads.
    
    Comparison with RAVLT backbone (GLTA):
    - RAVLT: 4-stage hierarchical, native multi-scale dims [64,128,256,512]
    - Ours:  flat ViT (d=256) + learned deconv/conv projections for multi-scale
    - RAVLT: alpha_j from attention scores
    - Ours:  w_j from MARL Actor-Critic (same purpose, RL-learned)
    """

    def __init__(self, patch_size=16, in_chans=3, d_model=256, depth=16,
                 num_heads=8, chunk_size=16, drop_path_rate=0.1,
                 out_channels=(256, 256, 256, 256),
                 fpn_scales=(2.0, 1.0, 0.5, 0.25),
                 out_indices=(3, 7, 11, 15),
                 freeze_router=False, init_cfg=None):
        if HAS_MMDET:
            super().__init__(init_cfg=init_cfg)
        else:
            super().__init__()

        self.d_model = d_model
        self.depth = depth
        self.out_indices = out_indices
        self.freeze_router = freeze_router

        # Patch Embedding
        self.patch_embed = PatchEmbedding(patch_size, in_chans, d_model)

        # Learnable positional embedding (interpolatable)
        self._default_H = 12
        self._default_W = 12
        self.pos_embed = nn.Parameter(torch.zeros(1, 144, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # Encoder Blocks
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, num_heads, chunk_size, drop_path=dpr[i])
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(d_model)

        # Shared Actor-Critic Router
        self.router = SharedActorCritic(d_model)

        # FPN Projection Layers
        self.fpn_projections = nn.ModuleList([
            FPNProjection(d_model, oc, sf)
            for oc, sf in zip(out_channels, fpn_scales)
        ])
        self.fpn_norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in out_indices])

    def _interpolate_pos_embed(self, H, W):
        N = H * W
        if N == self.pos_embed.shape[1]:
            return self.pos_embed
        pos = self.pos_embed.reshape(1, self._default_H, self._default_W, self.d_model)
        pos = pos.permute(0, 3, 1, 2)
        pos = F.interpolate(pos, size=(H, W), mode='bicubic', align_corners=False)
        return pos.permute(0, 2, 3, 1).reshape(1, N, self.d_model)

    def forward(self, x):
        tokens, H, W = self.patch_embed(x)
        tokens = tokens + self._interpolate_pos_embed(H, W)

        w_list, log_prob_list, value_list, mu_list, sigma_list = [], [], [], [], []
        fpn_features = {}
        deterministic = not self.training or self.freeze_router

        for i, block in enumerate(self.blocks):
            w, log_prob, value, mu, sigma = self.router(tokens, deterministic=deterministic)
            w_gating = torch.ones_like(w) if self.freeze_router else w
            tokens, _ = block(tokens, w_gating, H, W, use_dilution=(not self.freeze_router))

            w_list.append(w)
            log_prob_list.append(log_prob)
            value_list.append(value)
            mu_list.append(mu)
            sigma_list.append(sigma)

            if i in self.out_indices:
                fpn_features[i] = tokens

        tokens = self.norm(tokens)
        if self.out_indices[-1] == self.depth - 1:
            fpn_features[self.out_indices[-1]] = tokens

        outputs = []
        for idx, layer_idx in enumerate(self.out_indices):
            feat = self.fpn_norms[idx](fpn_features[layer_idx])
            outputs.append(self.fpn_projections[idx](feat, H, W))

        has_lp = log_prob_list[0] is not None
        self._routing_info = {
            'w_t': torch.stack(w_list, dim=1),
            'log_probs': torch.stack(log_prob_list, dim=1) if has_lp else None,
            'values': torch.stack(value_list, dim=1),
            'mu': torch.stack(mu_list, dim=1),
            'sigma': torch.stack(sigma_list, dim=1),
        }
        return tuple(outputs)


# Register with MMDetection
if HAS_MMDET:
    MODELS.register_module(name='MARLRALABackbone', module=MARLRALABackbone, force=True)
    print("[OK] MARLRALABackbone registered with MMDetection")
else:
    print("[OK] MARLRALABackbone defined (standalone mode)")

[OK] MARLRALABackbone registered with MMDetection


## 4. Reward Functions (No Explicit Budget)

In [22]:
import math
import torch
import torch.nn.functional as F
from torch.distributions import Normal

def compute_bimodal_sparsity(w_t):
    """
    Bimodal sparsity penalty: w * (1 - w)
    Pushes weights toward 0 or 1.
    """
    return (w_t * (1.0 - w_t)).mean()

def compute_ppo_reward(detection_loss, w_t, lambda_budget=2.0, lambda_sparse=1.0):
    """
    Combined PPO reward.
    
    reward = -det_loss - lambda_budget * mean(w_t) - lambda_sparse * bimodal_sparsity(w_t)
    
    Task loss pulls retention UP; budget penalty pushes it DOWN.
    Bimodal sparsity sharpens decisions towards 0/1.
    """
    task_reward = -detection_loss.detach()
    budget_cost = w_t.mean()
    sparsity_pen = compute_bimodal_sparsity(w_t)

    total = task_reward - lambda_budget * budget_cost - lambda_sparse * sparsity_pen

    return total, {
        'task_reward': task_reward.item(),
        'budget_cost': budget_cost.item(),
        'sparsity_pen': sparsity_pen.item(),
        'total_reward': total.item(),
        'w_mean': w_t.mean().item(),
        'w_std': w_t.std().item(),
    }


def ppo_update(router, routing_info, reward, optimizer,
               clip_eps=0.2, entropy_coeff=0.01, ppo_epochs=1, max_grad_norm=0.5):
    """PPO clipped surrogate update on router parameters only."""
    if routing_info['log_probs'] is None:
        return {'policy_loss': 0.0, 'skipped': True}

    old_log_probs = routing_info['log_probs'].detach()
    old_values = routing_info['values'].detach()
    w_t = routing_info['w_t'].detach()
    old_mu = routing_info['mu'].detach()
    old_sigma = routing_info['sigma'].detach()

    advantage = reward - old_values.mean()
    if advantage.numel() > 1:
        advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-8)

    total_pl, total_vl, total_ent = 0., 0., 0.
    for _ in range(ppo_epochs):
        z_old = torch.atanh((w_t * 2.0 - 1.0).clamp(-0.999, 0.999))
        dist_new = Normal(old_mu, old_sigma)
        new_log_probs = dist_new.log_prob(z_old)
        w_raw = torch.tanh(z_old)
        new_log_probs = new_log_probs - torch.log(0.5 * (1.0 - w_raw.pow(2)) + 1e-5)

        ratio = torch.exp(new_log_probs - old_log_probs)
        surr1 = ratio * advantage
        surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * advantage
        policy_loss = -torch.min(surr1, surr2).mean()
        value_loss = 0.5 * (reward - old_values).pow(2).mean()
        entropy = dist_new.entropy().mean()
        loss = policy_loss + 0.5 * value_loss - entropy_coeff * entropy

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(router.parameters(), max_grad_norm)
        optimizer.step()

        total_pl += policy_loss.item()
        total_vl += value_loss.item()
        total_ent += entropy.item()

    return {'policy_loss': total_pl/ppo_epochs, 'value_loss': total_vl/ppo_epochs,
            'entropy': total_ent/ppo_epochs}

print("[OK] Reward functions defined")


[OK] Reward functions defined


## 5. MMEngine Hooks

In [23]:
if HAS_MMDET:
    from mmengine.hooks import Hook

    @HOOKS.register_module(force=True)
    class FreezeRouterHook(Hook):
        """Freezes router for first N epochs, unfreezes for PPO."""
        def __init__(self, unfreeze_epoch=4):
            self.unfreeze_epoch = unfreeze_epoch

        def before_train_epoch(self, runner):
            model = runner.model.module if hasattr(runner.model, 'module') else runner.model
            backbone = model.backbone
            if runner.epoch < self.unfreeze_epoch:
                backbone.freeze_router = True
                for p in backbone.router.parameters(): p.requires_grad = False
                runner.logger.info(f"[FreezeRouterHook] Epoch {runner.epoch}: Router FROZEN")
            else:
                backbone.freeze_router = False
                for p in backbone.router.parameters(): p.requires_grad = True
                runner.logger.info(f"[FreezeRouterHook] Epoch {runner.epoch}: Router ACTIVE")


    @HOOKS.register_module(force=True)
    class PPORouterHook(Hook):
        """Runs PPO on the router after each training iteration."""
        def __init__(self, lambda_budget=2.0, lambda_sparse=1.0,
                     ppo_lr=1e-4, clip_eps=0.2, entropy_coeff=0.01,
                     ppo_epochs=1, log_interval=50):
            self.lambda_budget = lambda_budget
            self.lambda_sparse = lambda_sparse
            self.ppo_lr = ppo_lr
            self.clip_eps = clip_eps
            self.entropy_coeff = entropy_coeff
            self.ppo_epochs = ppo_epochs
            self.log_interval = log_interval
            self._optimizer = None

        def _get_optimizer(self, runner):
            if self._optimizer is None:
                model = runner.model.module if hasattr(runner.model, 'module') else runner.model
                self._optimizer = torch.optim.Adam(model.backbone.router.parameters(), lr=self.ppo_lr)
            return self._optimizer

        def after_train_iter(self, runner, batch_idx, data_batch=None, outputs=None):
            model = runner.model.module if hasattr(runner.model, 'module') else runner.model
            backbone = model.backbone
            if backbone.freeze_router: return
            if not hasattr(backbone, '_routing_info'): return
            ri = backbone._routing_info
            if ri.get('log_probs') is None: return
            if outputs is None or 'loss' not in outputs: return

            reward, comps = compute_ppo_reward(
                outputs['loss'].detach(), ri['w_t'],
                lambda_budget=self.lambda_budget,
                lambda_sparse=self.lambda_sparse)

            ppo_stats = ppo_update(
                backbone.router, ri, reward, self._get_optimizer(runner),
                self.clip_eps, self.entropy_coeff, self.ppo_epochs)

            if batch_idx % self.log_interval == 0:
                runner.logger.info(
                    f"[PPO] iter={batch_idx} | task_r={comps['task_reward']:.4f} | "
                    f"budget={comps['budget_cost']:.4f} | sparse={comps['sparsity_pen']:.4f} | "
                    f"w_mean={comps['w_mean']:.3f} | w_std={comps['w_std']:.3f}"
                )

    print("[OK] Hooks registered")
else:
    print("[SKIP] Hooks require MMDetection")


[OK] Hooks registered


## 6. Standalone Backbone Verification

In [26]:
print("=" * 60)
print("BACKBONE SHAPE TEST")
print("=" * 60)

model_test = MARLRALABackbone(
    patch_size=16, d_model=256, depth=16, num_heads=8, chunk_size=16,
    out_channels=(256, 256, 256, 256),
    fpn_scales=(2.0, 1.0, 0.5, 0.25),
    out_indices=(3, 7, 11, 15),
    freeze_router=True,
)

total = sum(p.numel() for p in model_test.parameters())
router_p = sum(p.numel() for n, p in model_test.named_parameters() if 'router' in n)
fpn_p = sum(p.numel() for n, p in model_test.named_parameters() if 'fpn' in n)
print(f"Total params:    {total:,}")
print(f"Router params:   {router_p:,}")
print(f"FPN proj params: {fpn_p:,}")
print(f"Backbone params: {total - router_p - fpn_p:,}")

for img_size in [224, 384, 512, 800]:
    try:
        x = torch.randn(1, 3, img_size, img_size)
        with torch.no_grad():
            outs = model_test(x)
        shapes = [f"{o.shape[2]}×{o.shape[3]}" for o in outs]
        print(f"  Input {img_size}×{img_size} → FPN levels: {shapes}")
    except Exception as e:
        print(f"  Input {img_size}×{img_size} → FAILED: {e}")

# Test unfrozen
model_test.freeze_router = False
model_test.train()
x = torch.randn(2, 3, 224, 224)
outs = model_test(x)
ri = model_test._routing_info
print(f"\nUnfrozen routing: w_mean={ri['w_t'].mean():.4f}, w_std={ri['w_t'].std():.4f}")
print(f"Log probs available: {ri['log_probs'] is not None}")

# Test reward
reward, comps = compute_ppo_reward(torch.tensor(5.0), ri['w_t'])
print(f"\nReward test:")
for k, v in comps.items():
    print(f"  {k}: {v:.4f}")

del model_test
print("\n" + "=" * 60)
print("ALL STANDALONE TESTS PASSED")
print("=" * 60)

BACKBONE SHAPE TEST
Total params:    16,239,875
Router params:   66,307
FPN proj params: 2,103,040
Backbone params: 14,070,528
  Input 224×224 → FPN levels: ['28×28', '14×14', '7×7', '4×4']
  Input 384×384 → FPN levels: ['48×48', '24×24', '12×12', '6×6']
  Input 512×512 → FPN levels: ['64×64', '32×32', '16×16', '8×8']
  Input 800×800 → FPN levels: ['100×100', '50×50', '25×25', '13×13']

Unfrozen routing: w_mean=0.4746, w_std=0.2681
Log probs available: True

Reward test:
  task_reward: -5.0000
  variance_bonus: 0.0703
  sparsity_penalty: 0.1775
  redundancy_penalty: 0.9657
  total_reward: -5.5198
  w_mean: 0.4746
  w_std: 0.2681

ALL STANDALONE TESTS PASSED


## 7. MMDetection Config (Programmatic)

This builds the same config as the original RALA repo's `retinanet_t_1x.py` but adapted for our backbone.

**Key differences from RALA repo config:**
- Backbone type: `MARLRALABackbone` instead of `GLTA`
- FPN in_channels: all 256 (projected) instead of `[64,128,256,512]` (native)
- Custom hooks: FreezeRouterHook + PPORouterHook
- Same optimizer, schedule, and head config as RALA repo

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — edit these paths for your system
# ═══════════════════════════════════════════════════════════════

COCO_ROOT = '/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017'
PRETRAINED_CKPT = '/kaggle/input/datasets/van2006/checkpt/vit_v2_stage_2_router_NEW_ARCHITECTURE.pth'  # or None for scratch
WORK_DIR = './work_dirs/marl_rala_retinanet_1x'
BATCH_SIZE = 8   # per GPU (matches RALA repo)
NUM_WORKERS = 2      # matches RALA repo
MAX_EPOCHS = 12      # 1× schedule
UNFREEZE_EPOCH = 4   # start PPO at epoch 4
USE_FP16 = True      # matches RALA repo (loss_scale=512)

print(f"COCO root: {COCO_ROOT}")
print(f"Pretrained: {PRETRAINED_CKPT}")
print(f"Work dir: {WORK_DIR}")
print(f"Schedule: {MAX_EPOCHS} epochs, unfreeze at {UNFREEZE_EPOCH}")

In [31]:
if HAS_MMDET:
    cfg_dict = dict(
        # ── Model (matches RALA repo RALA_retinanet.py structure) ──
        model=dict(
            type='RetinaNet',
            data_preprocessor=dict(
                type='DetDataPreprocessor',
                mean=[123.675, 116.28, 103.53], # Standard ImageNet means
                std=[58.395, 57.12, 57.375],    # Standard ImageNet stds
                bgr_to_rgb=True,                # OpenCV uses BGR, PyTorch uses RGB
                pad_size_divisor=32             # Critical: Pads images so FPN works
            ),
            backbone=dict(
                type='MARLRALABackbone',
                patch_size=12,
                in_chans=3,
                d_model=256,
                depth=16,
                num_heads=8,
                chunk_size=16,
                drop_path_rate=0.1,  # same as RALA repo
                out_channels=(256, 256, 256, 256),
                fpn_scales=(2.0, 1.0, 0.5, 0.25),
                out_indices=(3, 7, 11, 15),
                freeze_router=True,
                init_cfg=dict(type='Pretrained', checkpoint=PRETRAINED_CKPT)
                         if PRETRAINED_CKPT and os.path.exists(PRETRAINED_CKPT) else None,
            ),
            # FPN neck — same as RALA repo
            neck=dict(
                type='FPN',
                in_channels=[256, 256, 256, 256],
                out_channels=256,
                start_level=1,
                add_extra_convs='on_input',
                num_outs=5,
            ),
            # RetinaNet head — identical to RALA repo
            bbox_head=dict(
                type='RetinaHead',
                num_classes=80,
                in_channels=256,
                stacked_convs=4,
                feat_channels=256,
                anchor_generator=dict(
                    type='AnchorGenerator',
                    octave_base_scale=4,
                    scales_per_octave=3,
                    ratios=[0.5, 1.0, 2.0],
                    strides=[8, 16, 32, 64, 128]),
                bbox_coder=dict(
                    type='DeltaXYWHBBoxCoder',
                    target_means=[.0, .0, .0, .0],
                    target_stds=[1.0, 1.0, 1.0, 1.0]),
                loss_cls=dict(type='FocalLoss', use_sigmoid=True,
                              gamma=2.0, alpha=0.25, loss_weight=1.0),
                loss_bbox=dict(type='L1Loss', loss_weight=1.0)),
            train_cfg=dict(
                assigner=dict(type='MaxIoUAssigner', pos_iou_thr=0.5,
                              neg_iou_thr=0.4, min_pos_iou=0, ignore_iof_thr=-1),
                allowed_border=-1, pos_weight=-1, debug=False),
            test_cfg=dict(nms_pre=1000, min_bbox_size=0, score_thr=0.05,
                         nms=dict(type='nms', iou_threshold=0.5), max_per_img=100),
        ),

        # ── Dataset ──
        train_dataloader=dict(
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            persistent_workers=True,
            sampler=dict(type='DefaultSampler', shuffle=True),
            batch_sampler=dict(type='AspectRatioBatchSampler'),
            dataset=dict(
                type='CocoDataset',
                data_root=COCO_ROOT,
                ann_file='annotations/instances_train2017.json',
                data_prefix=dict(img='train2017/'),
                pipeline=[
                    dict(type='LoadImageFromFile'),
                    dict(type='LoadAnnotations', with_bbox=True),
                    dict(type='Resize', scale=(640, 640), keep_ratio=True),
                    dict(type='RandomFlip', prob=0.5),
                    dict(type='PackDetInputs'),
                ],
                filter_cfg=dict(filter_empty_gt=True, min_size=32),
            ),
        ),
        val_dataloader=dict(
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            persistent_workers=True,
            drop_last=False,
            sampler=dict(type='DefaultSampler', shuffle=False),
            dataset=dict(
                type='CocoDataset',
                data_root=COCO_ROOT,
                ann_file='annotations/instances_val2017.json',
                data_prefix=dict(img='val2017/'),
                pipeline=[
                    dict(type='LoadImageFromFile'),
                    dict(type='Resize', scale=(800, 1333), keep_ratio=True),
                    dict(type='LoadAnnotations', with_bbox=True),
                    dict(type='PackDetInputs',
                         meta_keys=('img_id', 'img_path', 'ori_shape',
                                    'img_shape', 'scale_factor')),
                ],
                test_mode=True,
            ),
        ),
        val_evaluator=dict(
            type='CocoMetric',
            ann_file=os.path.join(COCO_ROOT, 'annotations/instances_val2017.json'),
            metric='bbox',
        ),
        test_dataloader='${val_dataloader}',
        test_evaluator='${val_evaluator}',

        # ── Optimizer — matches RALA repo retinanet_t_1x.py ──
        optim_wrapper=dict(
            type='OptimWrapper',
            optimizer=dict(type='AdamW', lr=1e-4, weight_decay=1e-4),  # same as RALA repo
            clip_grad=dict(max_norm=1.0, norm_type=2),
        ),

        # ── Schedule: 1× — matches RALA repo schedule_1x.py ──
        param_scheduler=[
            dict(type='LinearLR', start_factor=0.001, by_epoch=False, begin=0, end=500),
            dict(type='MultiStepLR', begin=0, end=MAX_EPOCHS, by_epoch=True,
                 milestones=[8, 11], gamma=0.1),
        ],
        train_cfg=dict(type='EpochBasedTrainLoop', max_epochs=MAX_EPOCHS, val_interval=1),
        val_cfg=dict(type='ValLoop'),
        test_cfg=dict(type='TestLoop'),

        # ── Runtime ──
        default_scope='mmdet',
        default_hooks=dict(
            timer=dict(type='IterTimerHook'),
            logger=dict(type='LoggerHook', interval=50),
            param_scheduler=dict(type='ParamSchedulerHook'),
            checkpoint=dict(type='CheckpointHook', interval=1, max_keep_ckpts=1),
            sampler_seed=dict(type='DistSamplerSeedHook'),
            visualization=dict(type='DetVisualizationHook'),
        ),
        custom_hooks=[
            dict(type='FreezeRouterHook', unfreeze_epoch=UNFREEZE_EPOCH),
            dict(type='PPORouterHook', lambda_var=2.0, lambda_sparse=1.0,
                 lambda_redundancy=0.5, ppo_lr=1e-4, clip_eps=0.2,
                 entropy_coeff=0.01, ppo_epochs=1, log_interval=50),
        ],
        env_cfg=dict(
            cudnn_benchmark=False,
            mp_cfg=dict(mp_start_method='spawn', opencv_num_threads=0),
            dist_cfg=dict(backend='nccl'),
        ),
        vis_backends=[dict(type='LocalVisBackend')],
        visualizer=dict(type='DetLocalVisualizer', vis_backends=[dict(type='LocalVisBackend')],
                        name='visualizer'),
        log_processor=dict(type='LogProcessor', window_size=50, by_epoch=True),
        log_level='INFO',
        load_from=None,
        resume=False,
        work_dir=WORK_DIR,
    )

    cfg = Config(cfg_dict)
    print("[OK] MMDetection config built")
    print(f"  Model: {cfg.model.type}")
    print(f"  Backbone: {cfg.model.backbone.type}")
    print(f"  Epochs: {cfg.train_cfg.max_epochs}")
    print(f"  LR: {cfg.optim_wrapper.optimizer.lr}")
    print(f"  WD: {cfg.optim_wrapper.optimizer.weight_decay}")
else:
    print("[SKIP] Config requires MMDetection. Install first.")

[OK] MMDetection config built
  Model: RetinaNet
  Backbone: MARLRALABackbone
  Epochs: 12
  LR: 0.0001
  WD: 0.0001


## 8. Train (via MMEngine Runner)

In [ ]:
if HAS_MMDET:
    # Verify COCO paths exist
    for p, name in [
        (os.path.join(COCO_ROOT, 'train2017'), 'train images'),
        (os.path.join(COCO_ROOT, 'val2017'), 'val images'),
        (os.path.join(COCO_ROOT, 'annotations', 'instances_train2017.json'), 'train ann'),
        (os.path.join(COCO_ROOT, 'annotations', 'instances_val2017.json'), 'val ann'),
    ]:
        status = '[OK]' if os.path.exists(p) else '[MISSING]'
        print(f"  {status} {name}: {p}")
    
    print("\n--- Building Runner ---")
    runner = Runner.from_cfg(cfg)
    print("[OK] Runner built. Starting training...")
    runner.train()
    print("\n[DONE] Training complete!")
else:
    print("[SKIP] Training requires MMDetection")

  [OK] train images: /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/train2017
  [OK] val images: /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/val2017
  [OK] train ann: /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/instances_train2017.json
  [OK] val ann: /kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017/annotations/instances_val2017.json

--- Building Runner ---
04/12 12:20:07 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 270999706
    GPU 0,1: Tesla T4
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.8, V12.8.93
    GCC: x86_64-linux-gnu-gcc (Ubuntu 11.4.0-1ubuntu1~22.04.2) 11.4.0
    PyTorch: 2.10.0+cu128
    PyTorch compiling details: PyTorch built with:
  - GCC 13.3
  - C++ Version: 2017

/usr/local/lib/python3.12/dist-packages/mmengine/utils/manager.py:113: UserWarning: <class 'mmdet.visualization.local_visualizer.DetLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


04/12 12:20:07 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
04/12 12:20:07 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
(NORMAL      ) FreezeRouterHook                   
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_i

## 9. Evaluate

In [ ]:
if HAS_MMDET:
    # Find best checkpoint
    ckpt_dir = os.path.join(WORK_DIR)
    if os.path.isdir(ckpt_dir):
        ckpts = [f for f in os.listdir(ckpt_dir) if f.endswith('.pth')]
        if ckpts:
            latest = sorted(ckpts)[-1]
            print(f"Loading checkpoint: {latest}")
            cfg.load_from = os.path.join(ckpt_dir, latest)
            runner = Runner.from_cfg(cfg)
            metrics = runner.test()
            print("\n=== EVALUATION RESULTS ===")
            for k, v in metrics.items():
                print(f"  {k}: {v:.4f}")
        else:
            print("No checkpoints found. Train first.")
    else:
        print(f"Work dir not found: {ckpt_dir}")
else:
    print("[SKIP] Evaluation requires MMDetection")

## 10. Visualize Routing Maps

In [ ]:
import matplotlib.pyplot as plt

def visualize_routing(model, image_tensor, device='cuda'):
    """
    Visualize routing weights overlaid on the image.
    Shows which tokens the MARL router considers important.
    """
    model.eval()
    model.backbone.freeze_router = False
    
    with torch.no_grad():
        img = image_tensor.unsqueeze(0).to(device)
        _ = model.backbone(img)
        w_t = model.backbone._routing_info['w_t']  # (1, L, N)
    
    # Average across layers
    w_avg = w_t[0].mean(dim=0).cpu().numpy()  # (N,)
    
    # Reshape to spatial grid
    H_p = int(math.sqrt(len(w_avg)))
    W_p = H_p
    heatmap = w_avg.reshape(H_p, W_p)
    
    # Unnormalize image
    img_np = image_tensor.permute(1, 2, 0).cpu().numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_np = np.clip(std * img_np + mean, 0, 1)
    
    import cv2
    heatmap_resized = cv2.resize(heatmap, (img_np.shape[1], img_np.shape[0]),
                                  interpolation=cv2.INTER_CUBIC)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(img_np)
    axes[0].set_title('Input Image')
    axes[0].axis('off')
    
    im = axes[1].imshow(heatmap_resized, cmap='jet', vmin=0, vmax=1)
    axes[1].set_title(f'Routing Map\nmean={w_avg.mean():.3f}, std={w_avg.std():.3f}')
    axes[1].axis('off')
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    
    axes[2].imshow(img_np)
    axes[2].imshow(heatmap_resized, cmap='jet', alpha=0.4, vmin=0, vmax=1)
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Weight distribution
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(w_t[0].cpu().numpy().flatten(), bins=50, density=True, alpha=0.7)
    ax.set_xlabel('Routing Weight w')
    ax.set_ylabel('Density')
    ax.set_title('Distribution of Routing Weights (all layers)')
    ax.axvline(x=0.5, color='r', linestyle='--', label='0.5 center')
    ax.legend()
    plt.tight_layout()
    plt.show()

print("[OK] Visualization function defined")
print("Usage: visualize_routing(model, image_tensor, device='cuda')")

## 11. Baseline Config (No MARL — Fair Comparison)

Same backbone with `freeze_router=True` permanently (w=1.0 always, no PPO).
This is the control group — equivalent to running standard RALA without MARL routing.

In [ ]:
if HAS_MMDET:
    import copy
    baseline_cfg_dict = copy.deepcopy(cfg_dict)
    baseline_cfg_dict['model']['backbone']['freeze_router'] = True  # always frozen
    baseline_cfg_dict['custom_hooks'] = []  # no PPO, no freeze hook
    baseline_cfg_dict['work_dir'] = './work_dirs/rala_baseline_retinanet_1x'
    
    baseline_cfg = Config(baseline_cfg_dict)
    print("[OK] Baseline config built (w=1.0 always, no MARL)")
    print("  To train: runner = Runner.from_cfg(baseline_cfg); runner.train()")
else:
    print("[SKIP] Baseline config requires MMDetection")

## 12. Architecture Comparison Summary

### Original RALA Repo Structure
```
detection/
├── configs/
│   ├── _base_/
│   │   ├── models/RALA_retinanet.py     # backbone=GLTA, FPN, RetinaHead
│   │   ├── datasets/coco_detection.py
│   │   ├── schedules/schedule_1x.py     # SGD→AdamW overridden, steps=[8,11]
│   │   └── default_runtime.py
│   └── RALA/
│       ├── retinanet_t_1x.py            # RAVLT-T: [2,2,6,2], [64,128,256,512]
│       ├── maskrcnn_t_1x.py
│       ├── retinanet_s_1x.py            # RAVLT-S
│       └── ... (m, l variants)
└── backbone.py                          # GLTA backbone registration
```

### Our Structure (this notebook)
```
marl_vit_v2_det_mmdet.ipynb              # Everything in one notebook
├── Cell 2: Core modules (attention, router, encoder)
├── Cell 3: MARLRALABackbone + FPN projections
├── Cell 4: Reward functions (entropy, sparsity, variance)
├── Cell 5: MMEngine hooks (FreezeRouter, PPORouter)
├── Cell 7: Config (programmatic, matches RALA repo structure)
├── Cell 8: Training via MMEngine Runner
├── Cell 9: Evaluation
├── Cell 10: Routing visualization
└── Cell 11: Baseline config for fair comparison
```